In [7]:
import pandas as pd
import glob
import os

# 1. Point to the averages folder inside the cloned repo
data_dir = "../data/RABV_Pasteur_G_DMS/results/antibody_escape/averages/"

# 2. Find all the antibody CSV files
csv_files = glob.glob(os.path.join(data_dir, "*.csv"))
print(f"Found {len(csv_files)} antibody CSV files:")
for f in csv_files:
    print(os.path.basename(f))

# 3. Load one file (e.g., the first one) to inspect the columns
sample_file = csv_files[0]
print(f"\nInspecting: {os.path.basename(sample_file)}")
df = pd.read_csv(sample_file)

print("\nColumns:")
print(df.columns.tolist())
print("\nFirst 3 rows:")
print(df.head(3))

# 4. Load the site numbering map
map_path = "../data/RABV_Pasteur_G_DMS/data/site_numbering_map.csv"
site_map = pd.read_csv(map_path)

print("\nSite Numbering Map Columns:")
print(site_map.columns.tolist())
print("\nFirst 3 rows of Site Map:")
print(site_map.head(3))

Found 16 antibody CSV files:
17C7_mut_effect.csv
17C7_mut_icXX.csv
CR4098_mut_effect.csv
CR4098_mut_icXX.csv
CR57_mut_effect.csv
CR57_mut_icXX.csv
CTB012_mut_effect.csv
CTB012_mut_icXX.csv
RVA122_mut_effect.csv
RVA122_mut_icXX.csv
RVC20_mut_effect.csv
RVC20_mut_icXX.csv
RVC58_mut_effect.csv
RVC58_mut_icXX.csv
RVC68_mut_effect.csv
RVC68_mut_icXX.csv

Inspecting: 17C7_mut_effect.csv

Columns:
['epitope', 'site', 'wildtype', 'mutant', 'mutation', 'escape_mean', 'escape_median', 'escape_std', 'n_models', 'times_seen', 'frac_models', 'LibA-240319-17C7', 'LibB-240501-17C7']

First 3 rows:
   epitope  site wildtype mutant mutation  escape_mean  escape_median  \
0        1    -2        F      A     F-2A       0.3763         0.3763   
1        1    -2        F      C     F-2C      -0.1050        -0.1050   
2        1    -2        F      D     F-2D       0.1162         0.1162   

   escape_std  n_models  times_seen  frac_models  LibA-240319-17C7  \
0     0.48600         2       5.500          1.

In [8]:
!pip install --upgrade pyarrow

In [12]:
# ============================================================
# DAY 1: DATA ASSEMBLY — Build the Feature Matrix
# ============================================================
# Goal:
#   Create a unified, tidy dataframe with one row per
#   (antibody, mutation) pair, ready for downstream ML.
#
# Steps:
#   1. Load the site numbering map (region, reference sites)
#   2. Identify only the escape probability files (*_mut_effect.csv)
#   3. Tag each file with its antibody name
#   4. Concatenate all 8 antibodies into a single long dataframe
#   5. Merge with the site map for structural annotation
#   6. Save the final matrix as a Parquet file
#
# Output:
#   ../data/feature_matrix.parquet
# ============================================================

import pandas as pd
import glob
import os

# 1. Define paths
base_dir = "../data/RABV_Pasteur_G_DMS"
escape_dir = os.path.join(base_dir, "results/antibody_escape/averages/")
map_path = os.path.join(base_dir, "data/site_numbering_map.csv")

# 2. Load the site numbering map
site_map = pd.read_csv(map_path)

# 3. Get ONLY the '_mut_effect.csv' files (ignore the '_mut_icXX' ones)
effect_files = glob.glob(os.path.join(escape_dir, "*_mut_effect.csv"))

# 4. Loop through each file, add antibody name, and collect dataframes
dfs = []
for f in effect_files:
    # Extract antibody name from filename (e.g., "17C7" from "17C7_mut_effect.csv")
    ab_name = os.path.basename(f).replace("_mut_effect.csv", "")
    df = pd.read_csv(f)
    df["antibody"] = ab_name
    dfs.append(df)

# 5. Concatenate all 8 dataframes into one long matrix
feature_matrix = pd.concat(dfs, ignore_index=True)

# 6. Merge with the site map to get 'region' and 'reference_site'
# Note: The CSV uses 'site' for sequential_site, while the map uses 'sequential_site'
feature_matrix = feature_matrix.merge(
    site_map[['sequential_site', 'reference_site', 'region']],
    left_on='site',
    right_on='sequential_site',
    how='left'
)

# 7. Drop the redundant 'sequential_site' column from the map
feature_matrix = feature_matrix.drop(columns=['sequential_site'])

# 8. Inspect the final matrix
print(f"Feature matrix shape: {feature_matrix.shape}")
print("\nColumns:")
print(feature_matrix.columns.tolist())
print("\nFirst 3 rows:")
print(feature_matrix.head(3))

# 9. Save to parquet (with fallback to CSV if pyarrow still has issues)
try:
    feature_matrix.to_parquet("../data/feature_matrix.parquet", index=False)
    print("'feature_matrix.parquet' saved successfully.")
except Exception as e:
    print(f"\n⚠️ Parquet save failed: {e}")
    print("Falling back to CSV format...")
    feature_matrix.to_csv("../data/feature_matrix.csv", index=False)
    print("\n'feature_matrix.csv' saved successfully.")

Feature matrix shape: (65346, 30)

Columns:
['epitope', 'site', 'wildtype', 'mutant', 'mutation', 'escape_mean', 'escape_median', 'escape_std', 'n_models', 'times_seen', 'frac_models', 'LibA-240319-17C7', 'LibB-240501-17C7', 'antibody', 'LibA-240603-CR4098', 'LibB-240507-CR4098', 'LibA-240331-CR57', 'LibB-240510-CR57', 'LibA-240406-CTB012', 'LibB-240510-CTB012', 'LibA-240319-RVA122', 'LibB-240501-RVA122', 'LibA-240319-RVC20', 'LibB-240507-RVC20', 'LibA-240603-RVC58', 'LibB-240507-RVC58', 'LibA-240406-RVC68', 'LibB-240510-RVC68', 'reference_site', 'region']

First 3 rows:
   epitope  site wildtype mutant mutation  escape_mean  escape_median  \
0        1    -2        F      A     F-2A       0.3763         0.3763   
1        1    -2        F      C     F-2C      -0.1050        -0.1050   
2        1    -2        F      D     F-2D       0.1162         0.1162   

   escape_std  n_models  times_seen  ...  LibA-240319-RVA122  \
0     0.48600         2       5.500  ...                 NaN   
1

In [13]:
# ============================================================
# VERIFY: Is this really long format (one row per antibody-mutation)?
# ============================================================
import pandas as pd

df = pd.read_csv("../data/feature_matrix.csv")

print("Shape:", df.shape)
print("Unique antibodies:", df["antibody"].nunique())
print("Antibodies:", df["antibody"].unique())
print()

# Pick one mutation and look at its 8 rows
sample_mut = df["mutation"].iloc[0]
print(f"--- Rows for mutation {sample_mut} ---")
print(df[df["mutation"] == sample_mut][["antibody", "mutation", "escape_mean"]])

print()
# Pick 5 mutations randomly, count rows per mutation
print("--- Rows per mutation (first 5 mutations) ---")
print(df.groupby("mutation").size().head())

print()
# Does escape_mean vary across antibodies for the same mutation?
check = df[df["mutation"] == sample_mut]["escape_mean"]
print(f"escape_mean values for {sample_mut}: {check.tolist()}")
print(f"All identical? {check.nunique() == 1}  (should be False)")

Shape: (65346, 30)
Unique antibodies: 8
Antibodies: <StringArray>
['17C7', 'CR4098', 'CR57', 'CTB012', 'RVA122', 'RVC20', 'RVC58', 'RVC68']
Length: 8, dtype: str

--- Rows for mutation F-2A ---
      antibody mutation  escape_mean
0         17C7     F-2A     0.376300
8206    CR4098     F-2A     0.296400
16272     CR57     F-2A    -0.014410
24479   CTB012     F-2A    -0.006509
32709   RVA122     F-2A    -0.240000
40907    RVC20     F-2A     0.004527
49072    RVC58     F-2A    -0.008378
57203    RVC68     F-2A     0.051290

--- Rows per mutation (first 5 mutations) ---
mutation
A103C    8
A103D    8
A103E    8
A103F    8
A103G    8
dtype: int64

escape_mean values for F-2A: [0.3763, 0.2964, -0.01441, -0.006509, -0.24, 0.004527, -0.008378, 0.05129]
All identical? False  (should be False)


In [14]:
# ============================================================

# ============================================================
# Goal:
#   Establish the simplest possible predictors so we have a
#   floor to compare your ESM model against on Day 3.
#
# Steps:
#   1. Filter out signal peptide (antibodies can't bind there)
#   2. Binarize escape_mean into a classification target
#   3. Exp 1: Random predictor (the theoretical floor)
#   4. Exp 2: Sequence similarity predictor (physicochemical)
#   5. Exp 3: XGBoost on one-hot mutation + position
#   6. Report AUROC / AUPRC for all three
# ============================================================

import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings('ignore')

# ------------------------------------------------------------
# STEP 1: LOAD & FILTER DATA
# ------------------------------------------------------------
# Load the feature matrix we built on Day 1.
df = pd.read_csv('../data/feature_matrix.csv')
print('Original shape:', df.shape)

# Drop signal peptide rows. Antibodies cannot physically bind
# to the signal peptide, so keeping them adds noise.
df = df[df['region'] != 'signal_peptide'].copy()

# Drop any rows where escape_mean is missing (NaN).
df = df[df['escape_mean'].notna()].copy()
print('After filtering signal peptide + NaN:', df.shape)

# ------------------------------------------------------------
# STEP 2: DEFINE CLASSIFICATION TARGET
# ------------------------------------------------------------
# escape_mean is continuous. For a classifier we need a binary
# label. The standard DMS cutoff is > 0.5 (log-escape).
THRESHOLD = 0.5
df['label'] = (df['escape_mean'] > THRESHOLD).astype(int)
print('Positive rate:', df['label'].mean())

# Get the 8 unique antibodies for leave-one-out CV.
antibodies = sorted(df['antibody'].unique())
print('\nAntibodies:', antibodies)

# Helper function to compute AUROC and AUPRC.
def evaluate(y_true, y_scores, name):
    auroc = roc_auc_score(y_true, y_scores)
    auprc = average_precision_score(y_true, y_scores)
    print(f'  {name:25s} AUROC={auroc:.4f}  AUPRC={auprc:.4f}')
    return auroc, auprc

results = []

# ------------------------------------------------------------
# STEP 3: EXP 1 — RANDOM PREDICTOR
# ------------------------------------------------------------
# The floor. A model that cannot beat random is useless.
# We expect AUROC ~ 0.50 and AUPRC ~ = positive rate.
print('\n' + '='*60)
print('EXP 1: RANDOM PREDICTOR')
print('='*60)

all_y_true = []
all_y_score = []
for ab in antibodies:
    sub = df[df['antibody'] == ab]
    y_true = sub['label'].values
    np.random.seed(42)
    y_score = np.random.rand(len(y_true))  # random probabilities
    all_y_true.extend(y_true)
    all_y_score.extend(y_score)

evaluate(np.array(all_y_true), np.array(all_y_score), 'Random')
results.append(('Random',) + evaluate(np.array(all_y_true), np.array(all_y_score), 'Random'))

# ------------------------------------------------------------
# STEP 4: EXP 2 — SEQUENCE SIMILARITY PREDICTOR
# ------------------------------------------------------------
# Idea: mutations that are chemically similar to the wildtype
# should disrupt antibody binding less, so they should escape
# less. We use a simplified physicochemical distance (Grantham-like)
# between wildtype and mutant.
print('\n' + '='*60)
print('EXP 2: SEQUENCE SIMILARITY PREDICTOR')
print('='*60)

# Simplified physicochemical properties per amino acid.
# Values are (volume, polarity) approximations.
aa_props = {
    'A': (0.0, 0.0), 'R': (-1.0, 0.0), 'N': (0.0, -1.0), 'D': (1.0, -1.0),
    'C': (0.0, 1.0), 'Q': (0.0, -1.0), 'E': (1.0, -1.0), 'G': (0.0, 0.0),
    'H': (-0.5, 0.0), 'I': (-1.0, 1.0), 'L': (-1.0, 1.0), 'K': (-1.0, 0.0),
    'M': (-1.0, 1.0), 'F': (-1.0, 0.5), 'P': (0.0, 0.0), 'S': (0.0, -0.5),
    'T': (0.0, 0.5), 'W': (-1.0, 1.0), 'Y': (-1.0, 1.0), 'V': (-1.0, 1.0),
}

def sim_score(wt, mut):
    """Euclidean distance between wildtype and mutant properties."""
    if wt == mut:
        return 0.0
    if wt in aa_props and mut in aa_props:
        d = np.sqrt(sum((a - b) ** 2 for a, b in zip(aa_props[wt], aa_props[mut])))
        return d
    return 1.0  # default distance for unknown residues

all_y_true = []
all_y_score = []
for ab in antibodies:
    sub = df[df['antibody'] == ab].copy()
    # Compute similarity score for every mutation
    sub['sim'] = sub.apply(lambda r: sim_score(r['wildtype'], r['mutant']), axis=1)
    # Normalize to [0, 1] so it can act as a probability-like score
    sim_norm = (sub['sim'] - sub['sim'].min()) / (sub['sim'].max() - sub['sim'].min() + 1e-9)
    all_y_true.extend(sub['label'].values)
    all_y_score.extend(sim_norm.values)

evaluate(np.array(all_y_true), np.array(all_y_score), 'Sequence similarity')
results.append(('SeqSim',) + evaluate(np.array(all_y_true), np.array(all_y_score), 'SeqSim'))

# ------------------------------------------------------------
# STEP 5: EXP 3 — XGBOOST (ONE-HOT MUTATION + POSITION)
# ------------------------------------------------------------
# A proper ML baseline that does NOT use ESM.
# Features: one-hot encoding of (wildtype, mutant) pair + site position.
# Validation: Leave-One-Antibody-Out CV (train on 7, test on 1).
print('\n' + '='*60)
print('EXP 3: XGBOOST (one-hot mutation + position)')
print('='*60)

# Create a combined "wt_mut" string (e.g., "FA") and one-hot encode it.
df['wt_mut'] = df['wildtype'] + df['mutant']
enc = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
X_cat = enc.fit_transform(df[['wt_mut']])  # one-hot mutation features
X_pos = df[['site']].values.astype(float)  # raw site number
X = np.hstack([X_cat, X_pos])
y = df['label'].values

aucs = []
for ab in antibodies:
    # Hold out this antibody entirely
    mask_test = (df['antibody'] == ab).values
    X_train, X_test = X[~mask_test], X[mask_test]
    y_train, y_test = y[~mask_test], y[mask_test]

    # Skip if either split has only one class
    if len(np.unique(y_train)) < 2 or len(np.unique(y_test)) < 2:
        continue

    # Train XGBoost
    model = XGBClassifier(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.1,
        eval_metric='logloss',
        random_state=42
    )
    model.fit(X_train, y_train)

    # Predict probabilities on the held-out antibody
    y_score = model.predict_proba(X_test)[:, 1]

    auroc = roc_auc_score(y_test, y_score)
    auprc = average_precision_score(y_test, y_score)
    aucs.append((ab, auroc, auprc))
    print(f'  {ab:8s} AUROC={auroc:.4f}  AUPRC={auprc:.4f}')

print(f'\n  Mean XGBoost AUROC: {np.mean([a for _, a, _ in aucs]):.4f}')
print(f'  Mean XGBoost AUPRC: {np.mean([a for _, _, a in aucs]):.4f}')

# ------------------------------------------------------------
# STEP 6: SUMMARY TABLE
# ------------------------------------------------------------
# This is the table your ESM model must beat on Day 3.
print('\n' + '='*60)
print('SUMMARY')
print('='*60)
print(f'{"Baseline":<25} {"AUROC":<10} {"AUPRC":<10}')
print('-'*45)
for name, auroc, auprc in results:
    print(f'{name:<25} {auroc:<10.4f} {auprc:<10.4f}')
print(f'{"XGBoost (mean)":<25} {np.mean([a for _, a, _ in aucs]):<10.4f} {np.mean([a for _, _, a in aucs]):<10.4f}')

Original shape: (65346, 30)
After filtering signal peptide + NaN: (65034, 30)
Positive rate: 0.07097825752683212

Antibodies: ['17C7', 'CR4098', 'CR57', 'CTB012', 'RVA122', 'RVC20', 'RVC58', 'RVC68']

EXP 1: RANDOM PREDICTOR
  Random                    AUROC=0.4949  AUPRC=0.0707
  Random                    AUROC=0.4949  AUPRC=0.0707

EXP 2: SEQUENCE SIMILARITY PREDICTOR
  Sequence similarity       AUROC=0.5437  AUPRC=0.0792
  SeqSim                    AUROC=0.5437  AUPRC=0.0792

EXP 3: XGBOOST (one-hot mutation + position)
  17C7     AUROC=0.7045  AUPRC=0.1438
  CR4098   AUROC=0.7564  AUPRC=0.1288
  CR57     AUROC=0.6598  AUPRC=0.1039
  CTB012   AUROC=0.6562  AUPRC=0.1894
  RVA122   AUROC=0.7528  AUPRC=0.1458
  RVC20    AUROC=0.7038  AUPRC=0.1434
  RVC58    AUROC=0.7356  AUPRC=0.1391
  RVC68    AUROC=0.6555  AUPRC=0.1039

  Mean XGBoost AUROC: 0.7031
  Mean XGBoost AUPRC: 0.1373

SUMMARY
Baseline                  AUROC      AUPRC     
---------------------------------------------
Rando

In [15]:
import json
baseline_results = {
    'random':  {'auroc': 0.4949, 'auprc': 0.0707},
    'seqsim':  {'auroc': 0.5437, 'auprc': 0.0792},
    'xgboost': {'auroc': 0.7031, 'auprc': 0.1373},
}
with open('../data/baseline_results.json', 'w') as f:
    json.dump(baseline_results, f, indent=2)
print('✅ Baseline results saved.')

✅ Baseline results saved.


In [16]:
# Install ESM (fair-esm) and its dependencies
!pip install fair-esm

In [ ]:
# ============================================================

# ============================================================
# Goal:
#   Find the wildtype RABV-G protein sequence inside the
#   cloned DMS repo. We need it to extract 20-aa windows
#   around each mutation for ESM embedding.
# ============================================================

import os
import glob

base_dir = "../data/RABV_Pasteur_G_DMS"

# Search for common sequence file patterns
patterns = [
    "**/*.fasta", "**/*.fa", "**/*.faa", "**/*.fasta.gz",
    "**/*reference*", "**/*sequence*", "**/*.gb", "**/*.gbk"
]

print("=" * 60)
print("SEARCHING FOR SEQUENCE FILES")
print("=" * 60)
found = []
for p in patterns:
    hits = glob.glob(os.path.join(base_dir, p), recursive=True)
    for h in hits:
        if h not in found:
            found.append(h)

for f in found:
    size = os.path.getsize(f)
    print(f"  {os.path.relpath(f, base_dir)}  ({size} bytes)")

if not found:
    print("  No obvious sequence files found.")

# Also list the top-level 'data/' folder contents
print("\n" + "=" * 60)
print("CONTENTS OF data/ FOLDER")
print("=" * 60)
data_dir = os.path.join(base_dir, "data")
if os.path.isdir(data_dir):
    for item in sorted(os.listdir(data_dir)):
        full = os.path.join(data_dir, item)
        kind = "DIR " if os.path.isdir(full) else "FILE"
        print(f"  [{kind}] {item}")
else:
    print("  data/ folder not found.")

# Peek inside a likely reference file if one exists
print("\n" + "=" * 60)
print("FIRST FASTA-LIKE FILE (if any)")
print("=" * 60)
for f in found:
    if f.endswith((".fasta", ".fa", ".faa")):
        print(f"File: {os.path.relpath(f, base_dir)}")
        with open(f) as fh:
            for i, line in enumerate(fh):
                print("  " + line.rstrip())
                if i > 5:
                    break
        break
else:
    print("  No plain FASTA file found — we may need to look inside .zip or config files.")

SEARCHING FOR SEQUENCE FILES
  Additional_Data\AntibodySequences\RABV-G_DMS_AntibodySequences.fasta  (2214 bytes)
  Additional_Data\LibrarySpecs\Rabies-G_Pasteur-fullORF.fasta  (1640 bytes)
  Additional_Data\PrimerSequences\RABV-G_Mutagenesis-Primers.fasta  (1584 bytes)
  Additional_Data\PrimerSequences\RABV-G_Sequencing-Primers.fasta  (813 bytes)
  data\gene_sequence\codon.fasta  (1308 bytes)
  data\gene_sequence\protein.fasta  (442 bytes)
  non-pipeline_analyses\RABV_nextstrain\Results\G\nucleotide.fasta  (11585510 bytes)
  non-pipeline_analyses\RABV_nextstrain\Results\G\Alignments\codon.fasta  (11492161 bytes)
  non-pipeline_analyses\RABV_nextstrain\Results\G\Alignments\codon_ungapped.fasta  (11819819 bytes)
  non-pipeline_analyses\RABV_nextstrain\Results\G\Alignments\codon_ungapped_no_outgroup.fasta  (11818163 bytes)
  non-pipeline_analyses\RABV_nextstrain\Results\G\Alignments\protein.fasta  (4112733 bytes)
  non-pipeline_analyses\RABV_nextstrain\Results\G\Alignments\protein_ungapp

In [ ]:
# ============================================================

# ============================================================
# Goal:
#   Inspect the FASTA files so we know exactly what we're
#   working with before extracting ESM embeddings.
# ============================================================

from Bio import SeqIO

# --- 1. Load the RABV-G reference protein ---
print("=" * 60)
print("RABV-G REFERENCE PROTEIN")
print("=" * 60)
g_path = "../data/RABV_Pasteur_G_DMS/data/gene_sequence/protein.fasta"
for rec in SeqIO.parse(g_path, "fasta"):
    print(f"  ID:     {rec.id}")
    print(f"  Length: {len(rec.seq)} aa")
    print(f"  Seq:    {str(rec.seq)[:80]}...")

# --- 2. Load all antibody sequences ---
print("\n" + "=" * 60)
print("ANTIBODY SEQUENCES")
print("=" * 60)
ab_path = "../data/RABV_Pasteur_G_DMS/Additional_Data/AntibodySequences/RABV-G_DMS_AntibodySequences.fasta"
ab_seqs = {}
for rec in SeqIO.parse(ab_path, "fasta"):
    ab_seqs[rec.id] = str(rec.seq)
    print(f"  {rec.id:30s}  {len(rec.seq):4d} aa")

print(f"\nTotal sequences found: {len(ab_seqs)}")
print("IDs:", list(ab_seqs.keys()))

RABV-G REFERENCE PROTEIN
  ID:     gene
  Length: 433 aa
  Seq:    FGKFPIYTIPDKLGPWSPIDIHHLSCPNNLVVEDEGCTNLSGFSYMELKVGYISAIKMNGFTCTGVVTEAETYTNFVGYV...

ANTIBODY SEQUENCES
  CR4098_HeavyChain                119 aa
  CR4098_LightChain                107 aa
  17C7_HeavyChain                  119 aa
  17C7_LightChain                  107 aa
  CR57_HeavyChain                  127 aa
  CR57_LightChain                  112 aa
  RVA122_HeavyChain                123 aa
  RVA122_LightChain                110 aa
  RVC20_HeavyChain                 124 aa
  RVC20_LightChain                 107 aa
  RVC58_HeavyChain                 124 aa
  RVC58_LightChain                 110 aa
  RVC68_HeavyChain                 120 aa
  RVC68_LightChain                 107 aa
  CTB012_HeavyChain                120 aa
  CTB012_LightChain                112 aa

Total sequences found: 16
IDs: ['CR4098_HeavyChain', 'CR4098_LightChain', '17C7_HeavyChain', '17C7_LightChain', 'CR57_HeavyChain', 'CR57_LightChain', 'RVA12

In [ ]:
# ============================================================

# ============================================================
# Goal:
#   Run ESM-2 on the 433-aa G sequence ONCE, and cache
#   per-residue embeddings. Every mutation will reuse these.
#
# Model: esm2_t12_35M_UR50D (fast, 480-dim per residue)
#        → swap to esm2_t33_650M_UR50D later for 1280-dim.
# ============================================================

import torch
import esm
import numpy as np
from Bio import SeqIO
import os

# Detect device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

# --- Load G reference sequence ---
g_path = "../data/RABV_Pasteur_G_DMS/data/gene_sequence/protein.fasta"
g_seq = str(next(SeqIO.parse(g_path, "fasta")).seq)
print(f"G length: {len(g_seq)} aa")

# --- Load ESM-2 model ---
model, alphabet = esm.pretrained.esm2_t12_35M_UR50D()
model = model.eval().to(device)
batch_converter = alphabet.get_batch_converter()

# --- Run model on G ---
data = [("G", g_seq)]
_, _, tokens = batch_converter(data)
tokens = tokens.to(device)

with torch.no_grad():
    out = model(tokens, repr_layers=[12])

# Extract per-residue embeddings (drop BOS/EOS tokens)
emb = out["representations"][12][0, 1:len(g_seq)+1].cpu().numpy()
print(f"Embedding shape: {emb.shape}  (one vector per residue)")

# --- Save to disk so we don't recompute ---
os.makedirs("../data/esm_cache", exist_ok=True)
np.save("../data/esm_cache/G_esm_t12.npy", emb)
print("✅ Saved: ../data/esm_cache/G_esm_t12.npy")